# Final Base-Level EDA of Public NIfTI Scans

This notebook is the clean, presentation-level scan-level EDA for the public NIfTI dataset.

**Purpose:** verify that the scan files are readable, describe their spatial properties, summarize intensity distributions, and identify technical scan-level issues before mask-level analysis.

**Important:** this notebook should be run after `configs/datasets/public_dataset.json` has been created locally.


## 1. Setup and project paths

This cell finds the project root automatically. This avoids import errors when the notebook is opened from VS Code or Jupyter with a different working directory.


In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()

    start_path = Path(start_path).resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / "src").exists():
            return path

    raise FileNotFoundError("Could not find project root containing src/")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Current working directory:", Path.cwd())
print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())


## 2. Load configuration

The config file should point to the final scan folder used for both base-level and mask-level EDA.


In [ ]:
config_path = project_root / "configs" / "datasets" / "public_dataset.json"

if not config_path.exists():
    raise FileNotFoundError(
        f"Missing config file: {config_path}
"
        "Create this file locally with at least a 'public_root' key."
    )

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

scan_root = Path(config["public_root"])
mask_root = Path(config.get("totalseg_root", "")) if config.get("totalseg_root") else None

print("scan_root exists:", scan_root.exists(), scan_root)
if mask_root is not None:
    print("mask_root exists:", mask_root.exists(), mask_root)


## 3. Build or load scan inventory

Set `REGENERATE_INVENTORY = True` if you want to rebuild the inventory from the NIfTI files. If the output CSV already exists, the notebook loads it by default.


In [ ]:
from src.eda.build_public_scan_inventory import save_scan_inventory

output_dir = project_root / "metadata" / "eda" / "final_base_level"
output_dir.mkdir(parents=True, exist_ok=True)

scan_inventory_path = output_dir / "scan_inventory_current_public_root.csv"

REGENERATE_INVENTORY = False

if REGENERATE_INVENTORY or not scan_inventory_path.exists():
    scan_inventory_df = save_scan_inventory(
        root_dir=scan_root,
        output_csv=scan_inventory_path,
    )
else:
    scan_inventory_df = pd.read_csv(scan_inventory_path)

print("Inventory path:", scan_inventory_path)
print("Number of scans:", len(scan_inventory_df))
scan_inventory_df.head()


## 4. Dataset overview

This table gives the basic scan-level count and loading status.


In [ ]:
error_count = int(scan_inventory_df["error"].notna().sum()) if "error" in scan_inventory_df.columns else 0
unique_scan_ids = int(scan_inventory_df["scan_id"].nunique()) if "scan_id" in scan_inventory_df.columns else len(scan_inventory_df)

base_overview_df = pd.DataFrame([
    {"metric": "n_scans", "value": len(scan_inventory_df)},
    {"metric": "n_unique_scan_ids", "value": unique_scan_ids},
    {"metric": "n_loading_errors", "value": error_count},
])

base_overview_df


## 5. Spatial properties

This section summarizes the shape, slice count, spacing, field of view, and voxel count of the scans.


In [ ]:
shape_summary_df = (
    scan_inventory_df
    .groupby(["shape_x", "shape_y", "shape_z"], dropna=False)
    .size()
    .reset_index(name="n_scans")
    .sort_values("n_scans", ascending=False)
)

shape_summary_df


In [ ]:
spacing_cols = [col for col in ["spacing_x", "spacing_y", "spacing_z"] if col in scan_inventory_df.columns]

if spacing_cols:
    spacing_summary_df = (
        scan_inventory_df
        .groupby(spacing_cols, dropna=False)
        .size()
        .reset_index(name="n_scans")
        .sort_values("n_scans", ascending=False)
    )
else:
    spacing_summary_df = pd.DataFrame()

spacing_summary_df


In [ ]:
spatial_cols = [
    "shape_x", "shape_y", "shape_z",
    "spacing_x", "spacing_y", "spacing_z",
    "fov_x_mm", "fov_y_mm", "fov_z_mm",
    "num_voxels",
]

available_spatial_cols = [col for col in spatial_cols if col in scan_inventory_df.columns]
spatial_summary_df = scan_inventory_df[available_spatial_cols].describe().T
spatial_summary_df


## 6. Intensity properties

This section summarizes global scan intensity statistics. These values are useful for checking whether scans have extreme intensity ranges or unusual nonzero fractions.


In [ ]:
intensity_cols = [
    "min_intensity",
    "max_intensity",
    "mean_intensity",
    "std_intensity",
    "median_intensity",
    "p01_intensity",
    "p05_intensity",
    "p95_intensity",
    "p99_intensity",
    "nonzero_fraction",
]

available_intensity_cols = [col for col in intensity_cols if col in scan_inventory_df.columns]
intensity_summary_df = scan_inventory_df[available_intensity_cols].describe().T
intensity_summary_df


## 7. Scan-level QC flags

These flags are conservative technical checks. They do not automatically remove scans; they mark scans that should be inspected if they affect later analysis.


In [ ]:
def iqr_bounds(series, multiplier=1.5):
    series = pd.to_numeric(series, errors="coerce").dropna()
    if len(series) == 0:
        return None, None
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - multiplier * iqr, q3 + multiplier * iqr


def make_scan_qc_flags(df):
    rows = []

    valid_df = df.copy()
    if "error" in valid_df.columns:
        valid_df = valid_df[valid_df["error"].isna()].copy()

    modal_shape_z = valid_df["shape_z"].mode().iloc[0] if "shape_z" in valid_df.columns and len(valid_df) else None

    spacing_cols_local = [col for col in ["spacing_x", "spacing_y", "spacing_z"] if col in valid_df.columns]
    modal_spacing = None
    if spacing_cols_local and len(valid_df):
        modal_spacing = tuple(valid_df[spacing_cols_local].mode().iloc[0].tolist())

    mean_low, mean_high = iqr_bounds(valid_df["mean_intensity"]) if "mean_intensity" in valid_df.columns else (None, None)
    nonzero_low, nonzero_high = iqr_bounds(valid_df["nonzero_fraction"]) if "nonzero_fraction" in valid_df.columns else (None, None)

    for _, row in df.iterrows():
        flags = []

        if "error" in df.columns and pd.notna(row.get("error")):
            flags.append("loading_error")

        if "num_dimensions" in df.columns and pd.notna(row.get("num_dimensions")) and int(row["num_dimensions"]) != 3:
            flags.append("not_3d")

        if modal_shape_z is not None and pd.notna(row.get("shape_z")) and row["shape_z"] != modal_shape_z:
            flags.append("non_modal_slice_count")

        if modal_spacing is not None:
            row_spacing = tuple(row[col] for col in spacing_cols_local)
            if row_spacing != modal_spacing:
                flags.append("non_modal_spacing")

        if mean_low is not None and pd.notna(row.get("mean_intensity")):
            if row["mean_intensity"] < mean_low or row["mean_intensity"] > mean_high:
                flags.append("mean_intensity_outlier")

        if nonzero_low is not None and pd.notna(row.get("nonzero_fraction")):
            if row["nonzero_fraction"] < nonzero_low or row["nonzero_fraction"] > nonzero_high:
                flags.append("nonzero_fraction_outlier")

        rows.append({
            "file_name": row.get("file_name"),
            "scan_id": row.get("scan_id"),
            "qc_status": "check" if flags else "ok",
            "qc_flags": ",".join(flags),
        })

    return pd.DataFrame(rows)


scan_qc_flags_df = make_scan_qc_flags(scan_inventory_df)
scan_qc_flags_df[scan_qc_flags_df["qc_status"] != "ok"].head(20)


In [ ]:
scan_qc_counts_df = (
    scan_qc_flags_df
    .assign(qc_flags_split=scan_qc_flags_df["qc_flags"].str.split(","))
    .explode("qc_flags_split")
)
scan_qc_counts_df = scan_qc_counts_df[scan_qc_counts_df["qc_flags_split"].notna() & (scan_qc_counts_df["qc_flags_split"] != "")]
scan_qc_counts_df = scan_qc_counts_df.groupby("qc_flags_split").size().reset_index(name="n_scans")
scan_qc_counts_df


## 8. Final base-level figures

These are the main figures to use in the EDA report. They are also saved to `docs/figures/final_base_level/`.


In [ ]:
figure_dir = project_root / "docs" / "figures" / "final_base_level"
figure_dir.mkdir(parents=True, exist_ok=True)

print("Figure directory:", figure_dir)


In [ ]:
plt.figure(figsize=(8, 4))
shape_labels = shape_summary_df.apply(
    lambda row: f"{int(row['shape_x'])}x{int(row['shape_y'])}x{int(row['shape_z'])}" if pd.notna(row['shape_x']) else "missing",
    axis=1,
)
plt.bar(shape_labels, shape_summary_df["n_scans"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of scans")
plt.title("Scan shape distribution")
plt.tight_layout()
plt.savefig(figure_dir / "scan_shape_distribution.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
slice_counts = scan_inventory_df["shape_z"].value_counts(dropna=False).sort_index()
plt.bar(slice_counts.index.astype(str), slice_counts.values)
plt.xlabel("Number of slices")
plt.ylabel("Number of scans")
plt.title("Slice count distribution")
plt.tight_layout()
plt.savefig(figure_dir / "slice_count_distribution.png", dpi=150)
plt.show()


In [ ]:
if len(spacing_summary_df) > 0:
    plt.figure(figsize=(8, 4))
    spacing_labels = spacing_summary_df.apply(
        lambda row: " x ".join(str(row[col]) for col in spacing_cols),
        axis=1,
    )
    plt.bar(spacing_labels, spacing_summary_df["n_scans"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Number of scans")
    plt.title("Voxel spacing distribution")
    plt.tight_layout()
    plt.savefig(figure_dir / "spacing_distribution.png", dpi=150)
    plt.show()
else:
    print("No spacing columns available.")


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(scan_inventory_df["mean_intensity"].dropna(), bins=30)
plt.xlabel("Mean intensity")
plt.ylabel("Number of scans")
plt.title("Mean intensity distribution")
plt.tight_layout()
plt.savefig(figure_dir / "mean_intensity_distribution.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(scan_inventory_df["nonzero_fraction"].dropna(), bins=30)
plt.xlabel("Nonzero fraction")
plt.ylabel("Number of scans")
plt.title("Nonzero fraction distribution")
plt.tight_layout()
plt.savefig(figure_dir / "nonzero_fraction_distribution.png", dpi=150)
plt.show()


## 9. Save final base-level EDA outputs


In [ ]:
base_overview_df.to_csv(output_dir / "base_overview.csv", index=False)
shape_summary_df.to_csv(output_dir / "shape_summary.csv", index=False)
spacing_summary_df.to_csv(output_dir / "spacing_summary.csv", index=False)
spatial_summary_df.to_csv(output_dir / "spatial_summary.csv")
intensity_summary_df.to_csv(output_dir / "intensity_summary.csv")
scan_qc_flags_df.to_csv(output_dir / "scan_qc_flags.csv", index=False)
scan_qc_counts_df.to_csv(output_dir / "scan_qc_flag_counts.csv", index=False)

print("Saved final base-level EDA tables to:", output_dir)


## 10. Report-ready interpretation

Use this short interpretation as the basis for the thesis/report text.


In [ ]:
n_scans = len(scan_inventory_df)
n_errors = error_count
n_shapes = len(shape_summary_df)
modal_shape = shape_summary_df.iloc[0]
modal_shape_text = f"{int(modal_shape['shape_x'])} x {int(modal_shape['shape_y'])} x {int(modal_shape['shape_z'])}"

if len(spacing_summary_df) > 0:
    modal_spacing_row = spacing_summary_df.iloc[0]
    modal_spacing_text = " x ".join(str(modal_spacing_row[col]) for col in spacing_cols)
else:
    modal_spacing_text = "not available"

base_interpretation = f"""
Base-level EDA summary:
The final scan inventory contains {n_scans} NIfTI scans. The loading-error count is {n_errors}. 
The most common scan shape is {modal_shape_text}, and the most common voxel spacing is {modal_spacing_text}. 
The scan-level EDA confirms that the scans are technically usable for downstream analysis, while also showing that spatial dimensions and intensity properties should be handled explicitly during preprocessing. 
The scan inventory matches the dataset version used for the TotalSegmentator mask-level EDA.
"""

print(base_interpretation)


## Final base-level EDA conclusion

The scan-level EDA confirms that the public NIfTI scans can be inventoried and summarized consistently. The final scan count should match the mask-level EDA scan count. This base-level EDA is used to justify that downstream organ-level analysis is performed on a technically consistent and traceable dataset version.
